# Task 1: Data Cleaning & Preprocessing — Titanic Dataset

## Problem Statement
Raw datasets almost never come ready for analysis. They have missing values, duplicates, wrong data types, and inconsistent column names. Before any meaningful analysis or modeling can happen, the data must be cleaned.

In this notebook, I clean the Titanic dataset by:
1. Handling missing values
2. Removing duplicates
3. Converting data types
4. Renaming columns for clarity

**Output:** A clean dataset (`titanic_cleaned.csv`) ready for EDA and modeling.

**Dataset source:** [Kaggle — Titanic: Machine Learning from Disaster](https://www.kaggle.com/c/titanic/data)

## Step 1: Import libraries and load the dataset

In [3]:
import pandas as pd # import pandas
import numpy as np

In [4]:
df = pd.read_csv("train.csv") # Loads the dataset CSV into a DataFrame
print(df)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

In [5]:
# Keep a copy of the raw data for comparison later
df_raw = df.copy()
print(df_raw) # printing copy

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

In [6]:
print(f"Dataset Loaded. Shape: {df.shape}") #df.shape returns (rows,columns)

Dataset Loaded. Shape: (891, 12)


In [7]:
df.head()  # df.head() showsfirst 6 rows, df.head(5) shows first 5 rows

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
df.info() # Column data types and non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [9]:
# df.isnull().sum() # Counts missing values per column
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum()/len(df)*100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Mising %': missing_pct
})

missing_summary[missing_summary['Missing Count'] > 0]

,Missing Count,Mising %
Age,177,19.87
Cabin,687,77.10
Embarked,2,0.22


### Observations

- **Age** — 177 missing values (~19.9%). Important feature, can't drop. Will **impute** with median.
- **Cabin** — 687 missing values (~77.1%). Too many missing to impute meaningfully. Will **drop** the column, but first extract the *deck letter* as a new feature where available.
- **Embarked** — only 2 missing values (~0.2%). Will **impute** with mode (most frequent value).
- No duplicate rows (verified below).

### Why these specific strategies?
- **Median over mean for Age**: Age has outliers (a few elderly passengers), and median is more robust to outliers than mean.
- **Mode for Embarked**: It's categorical, so mean/median doesn't apply. The most frequent port (`S` = Southampton) is a safe default for 2 missing rows.
- **Drop Cabin**: With 77% missing, imputation would invent data. But the deck letter (first character) is useful, so we extract it first.

In [10]:
print(type(df))

<class 'pandas.DataFrame'>


## Step 3: Check for duplicates

In [18]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

# Remove duplicates if any exist (defensive — keeps notebook reusable on other datasets)
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

Number of duplicate rows: 0
Shape after removing duplicates: (891, 12)


## Step 4: Handle missing values

### 4a. Extract deck letter from Cabin, then drop Cabin

In [20]:
# Cabin values look like 'C85', 'E46', 'C23 C25 C27' — first letter = deck.
# Create a new 'Deck' column. Missing cabins become 'Unknown'.
df['Deck'] = df['Cabin'].astype(str).str[0]
df.loc[df['Cabin'].isnull(), 'Deck'] = 'Unknown'

print("Deck value counts:")
print(df['Deck'].value_counts())

# Now drop the original Cabin column
df = df.drop(columns=['Cabin'])

Deck value counts:
Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64


### 4b. Impute missing Age with median

In [21]:
median_age = df['Age'].median()
print(f"Median age: {median_age}")

df['Age'] = df['Age'].fillna(median_age)
print(f"Missing values in Age after imputation: {df['Age'].isnull().sum()}")

Median age: 28.0
Missing values in Age after imputation: 0


### 4c. Impute missing Embarked with mode

In [22]:
mode_embarked = df['Embarked'].mode()[0]
print(f"Most frequent Embarked value: {mode_embarked}")

df['Embarked'] = df['Embarked'].fillna(mode_embarked)
print(f"Missing values in Embarked after imputation: {df['Embarked'].isnull().sum()}")

Most frequent Embarked value: S
Missing values in Embarked after imputation: 0


## Step 5: Convert data types

Pandas infers types when loading a CSV, but some columns are better represented differently:
- **Survived, Pclass, Sex, Embarked, Deck** — these are categorical, not continuous. Converting them to `category` dtype saves memory and makes intent clearer.
- **Age** — after imputation it can be cast to `int` (whole years are fine for Titanic analysis).

In [23]:
# Convert categorical columns
categorical_cols = ['Survived', 'Pclass', 'Sex', 'Embarked', 'Deck']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# Convert Age to integer
df['Age'] = df['Age'].astype(int)

print("Updated data types:")
print(df.dtypes)

Updated data types:
PassengerId       int64
Survived       category
Pclass         category
Name                str
Sex            category
Age               int64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Embarked       category
Deck           category
dtype: object


## Step 6: Rename columns

Original column names use abbreviations and mixed casing. I'll convert them to clear, snake_case names — a common convention in data analysis.

In [24]:
df = df.rename(columns={
    'PassengerId': 'passenger_id',
    'Survived': 'survived',
    'Pclass': 'passenger_class',
    'Name': 'name',
    'Sex': 'sex',
    'Age': 'age',
    'SibSp': 'siblings_spouses_aboard',
    'Parch': 'parents_children_aboard',
    'Ticket': 'ticket',
    'Fare': 'fare',
    'Embarked': 'embarked_port',
    'Deck': 'deck'
})

print("Renamed columns:")
print(df.columns.tolist())

Renamed columns:
['passenger_id', 'survived', 'passenger_class', 'name', 'sex', 'age', 'siblings_spouses_aboard', 'parents_children_aboard', 'ticket', 'fare', 'embarked_port', 'deck']


## Step 7: Final inspection of the cleaned dataset

In [25]:
df.head()

,passenger_id,survived,passenger_class,name,sex,age,siblings_spouses_aboard,parents_children_aboard,ticket,fare,embarked_port,deck
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,S,Unknown
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,S,Unknown
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1000,S,C
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,S,Unknown


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   passenger_id             891 non-null    int64   
 1   survived                 891 non-null    category
 2   passenger_class          891 non-null    category
 3   name                     891 non-null    str     
 4   sex                      891 non-null    category
 5   age                      891 non-null    int64   
 6   siblings_spouses_aboard  891 non-null    int64   
 7   parents_children_aboard  891 non-null    int64   
 8   ticket                   891 non-null    str     
 9   fare                     891 non-null    float64 
 10  embarked_port            891 non-null    category
 11  deck                     891 non-null    category
dtypes: category(5), float64(1), int64(4), str(2)
memory usage: 53.4 KB


In [27]:
df.describe(include='all')

,passenger_id,survived,passenger_class,name,sex,age,siblings_spouses_aboard,parents_children_aboard,ticket,fare,embarked_port,deck
count,891.000000,891.0,891.0,891,891,891.000000,891.000000,891.000000,891,891.000000,891,891
unique,NaN,2.0,3.0,891,2,NaN,NaN,NaN,681,NaN,3,9
top,NaN,0.0,3.0,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,S,Unknown
freq,NaN,549.0,491.0,1,577,NaN,NaN,NaN,7,NaN,646,687
mean,446.000000,NaN,NaN,NaN,NaN,29.345679,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,NaN,NaN,NaN,NaN,13.028212,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,NaN,NaN,NaN,NaN,22.000000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,NaN,NaN,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,NaN,NaN,NaN,NaN,35.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


## Step 8: Before vs. After Summary

In [28]:
summary = pd.DataFrame({
    'Before Cleaning': [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum()
    ],
    'After Cleaning': [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
}, index=['Rows', 'Columns', 'Total Missing Values', 'Duplicate Rows'])

summary

,Before Cleaning,After Cleaning
Rows,891,891
Columns,12,12
Total Missing Values,866,0
Duplicate Rows,0,0


## Step 9: Save the cleaned dataset

In [30]:
df.to_csv('titanic_cleaned.csv', index=False)
print("Cleaned dataset saved as 'titanic_cleaned.csv'")
print(f"Final shape: {df.shape}")

Cleaned dataset saved as 'titanic_cleaned.csv'
Final shape: (891, 12)


## Conclusion

The Titanic dataset has been successfully cleaned and preprocessed:

- ✅ **Missing values handled**: Age imputed with median, Embarked imputed with mode, Cabin dropped (with deck letter preserved as a new feature)
- ✅ **Duplicates checked**: None found in this dataset
- ✅ **Data types corrected**: Categorical columns converted to `category` dtype; Age cast to integer
- ✅ **Columns renamed**: All columns in clear snake_case
- ✅ **Output saved**: `titanic_cleaned.csv` is ready for EDA and modeling

### Key insights from the cleaning process
1. About 1 in 5 age values were missing — imputation was necessary, not optional.
2. The Cabin column was 77% empty, so extracting just the deck letter preserved the signal without inventing data.
3. The dataset had no exact duplicate rows, which is typical for well-curated Kaggle datasets — but the check should always be run regardless.

### Next steps
This cleaned dataset will feed into Task 2 (Data Visualization) and Task 3 (Exploratory Data Analysis).